<a href="https://colab.research.google.com/github/Didier06/IA_licence_pro_chimie/blob/main/MLP_GNN/Toxicite_Keras.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Prédiction de Toxicité avec Keras et RDKit
Dans ce notebook, nous allons construire un réseau de neurones avec Keras (TensorFlow) pour classifier des molécules (Toxique vs Non-Toxique) en utilisant les empreintes digitales de Morgan (Morgan Fingerprints).

In [4]:
# Installation des dépendances (décommentez et exécutez si nécessaire)
#!pip install rdkit tensorflow scikit-learn numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.0/37.0 MB 48.6 MB/s eta 0:00:00


In [5]:
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem
from sklearn.model_selection import train_test_split
from tensorflow import keras
from tensorflow.keras import layers

# 1. DONNÉES D'EXEMPLE
# 1 = Toxique, 0 = Non toxique
data = [
    ("c1ccccc1", 1),       # Benzène
    ("CCO", 0),            # Éthanol
    ("C(=O)O", 0),         # Acide formique
    ("C1=CC=C(C=C1)O", 1), # Phénol
    ("C", 0),              # Méthane
    ("C1=CC=C(C=C1)Cl", 1) # Chlorobenzène
]

smiles_list = [item[0] for item in data]
labels = [item[1] for item in data]

In [6]:
# 2. CONVERSION EN VECTEURS (Morgan Fingerprints)
def smiles_to_fingerprint(smiles, radius=2, n_bits=2048):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return np.zeros((n_bits,))
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
    return np.array(fp)

X = np.array([smiles_to_fingerprint(s) for s in smiles_list])
y = np.array(labels)

# 3. SÉPARATION ENTRAÎNEMENT / TEST
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

print(f"Taille des données d'entraînement : {X_train.shape}")
print(f"Taille des données de test : {X_test.shape}")

Taille des données d'entraînement : (4, 2048)
Taille des données de test : (2, 2048)


[09:37:01] DEPRECATION WARNING: please use MorganGenerator
[09:37:01] DEPRECATION WARNING: please use MorganGenerator
[09:37:01] DEPRECATION WARNING: please use MorganGenerator
[09:37:01] DEPRECATION WARNING: please use MorganGenerator
[09:37:01] DEPRECATION WARNING: please use MorganGenerator
[09:37:01] DEPRECATION WARNING: please use MorganGenerator


In [7]:
# 4. CRÉATION DU MODÈLE KERAS
model = keras.Sequential([
    # Couche d'entrée qui correspond à la taille de notre fingerprint (2048)
    keras.Input(shape=(2048,)),

    # Première couche cachée avec 64 neurones et activation ReLU
    layers.Dense(64, activation='relu'),

    # Couche de régularisation (Dropout) pour éviter le surapprentissage
    layers.Dropout(0.2),

    # Deuxième couche cachée avec 32 neurones
    layers.Dense(32, activation='relu'),

    # Couche de sortie : 1 neurone avec activation Sigmoid pour une probabilité (0 à 1)
    layers.Dense(1, activation='sigmoid')
])

# Compilation du modèle
# On utilise binary_crossentropy car c'est un problème de classification binaire
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │       131,136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 133,249 (520.50 KB)

 Trainable params: 133,249 (520.50 KB)

 Non-trainable params: 0 (0.00 B)

In [8]:
# 5. ENTRAÎNEMENT DU MODÈLE
print("Début de l'entraînement...")
history = model.fit(
    X_train, y_train,
    epochs=50,           # Nombre de passages sur l'ensemble des données
    batch_size=2,        # Mise à jour des poids toutes les 2 molécules
    verbose=1,
    validation_data=(X_test, y_test)
)
print("Entraînement terminé !")

Début de l'entraînement...
Epoch 1/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 362ms/step - accuracy: 0.7500 - loss: 0.6923 - val_accuracy: 0.5000 - val_loss: 0.6874
Epoch 2/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step - accuracy: 1.0000 - loss: 0.6812 - val_accuracy: 0.5000 - val_loss: 0.6830
Epoch 3/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step - accuracy: 0.7500 - loss: 0.6725 - val_accuracy: 1.0000 - val_loss: 0.6785
Epoch 4/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step - accuracy: 1.0000 - loss: 0.6527 - val_accuracy: 1.0000 - val_loss: 0.6743
Epoch 5/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step - accuracy: 1.0000 - loss: 0.6336 - val_accuracy: 1.0000 - val_loss: 0.6707
Epoch 6/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step - accuracy: 1.0000 - loss: 0.6331 - val_accuracy: 1.0000 - val_loss: 0.6673
Epoch 7/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step - accuracy: 1.0000 - loss: 0.6503 - val_accuracy: 1.0000 - val_loss: 0.6639
Epoch 8/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step - accuracy: 1.0000 - loss: 0.6163 - val_accuracy

In [9]:
# 6. ÉVALUATION ET PRÉDICTION SUR UNE NOUVELLE MOLÉCULE
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Précision sur le jeu de test : {accuracy * 100:.2f}%\n")

nouvelle_molecule_smiles = "c1ccccc1C" # Toluène
nouvelle_empreinte = smiles_to_fingerprint(nouvelle_molecule_smiles)
nouvelle_empreinte = nouvelle_empreinte.reshape(1, -1) # Keras attend (batch_size, n_features)

# Prédire la probabilité (entre 0 et 1)
probabilite = model.predict(nouvelle_empreinte, verbose=0)[0][0]

print(f"--- Analyse du Toluène ({nouvelle_molecule_smiles}) ---")
if probabilite >= 0.5:
    print(f"Prédiction : TOXIQUE (Confiance : {probabilite*100:.2f}%)")
else:
    print(f"Prédiction : NON TOXIQUE (Confiance : {(1-probabilite)*100:.2f}%)")

Précision sur le jeu de test : 100.00%

--- Analyse du Toluène (c1ccccc1C) ---
Prédiction : TOXIQUE (Confiance : 89.42%)


[09:37:22] DEPRECATION WARNING: please use MorganGenerator
